# Module 1.4 — Reflection & Reflexion: Self-Improving Agents

**Where we are.** Notebook 4 gave the agent MCP tools (arXiv search, Ising Monte Carlo). You also saw the characteristic failure mode: the agent confidently returned a list of raw JSON blobs when the task asked for a plain-English description. Small local models (7–8 B) are competent at the tool-call loop but lazy about follow-through. This notebook gives the agent two mechanisms to catch its own mistakes.

**Two ideas, one notebook.**

1. **Reflection** — *within-task* self-correction. After the agent produces a final answer, a second LLM call (the "Critic") reads the task and the answer together and either signs off or demands a revision. A minimal wrapper loops: *actor → critic → actor-with-feedback → critic …* for up to 2 rounds.
2. **Reflexion** (with an *x*; Shinn et al., 2023) — *across-task* learning. Every time the Critic rejects an answer, we distil the lesson and append it to a JSON file on disk. Before the next task, those past lessons are prepended to the prompt. Over a session, the agent acquires a little "logbook" of what it has been told.

**Physicist's bridge.**

| Agent idea | What it reminds you of |
|---|---|
| Reflection | Iterative numerical methods: solve, compute residual, correct. |
| Reflexion  | Experimental logbook: record what worked and what didn't, *read it next time*. |

Neither is deep — both are just loops around an LLM call. The pedagogical point is that *the model itself does not change*; what changes is the *scaffolding* around it. That's the hallmark of modern agent engineering.

**What we'll demo.** The outline's finite-size-effect story:
- **Task 1** — "Estimate T_c on an 8×8 lattice." The agent does a scan, finds the specific-heat peak somewhere near 2.35–2.45, and reports it. The Critic notices the answer is ≥3% above Onsager's exact 2.269 and flags finite-size effects. The agent revises, pointing out the shift and the need for a larger lattice.
- **Task 2** — "Estimate T_c again, but this time acknowledge what you learned." The agent (now carrying the Reflexion memory) uses L = 32 from the start.
- **Task 3** — A short theoretical-exponent question that only makes sense once the lesson is remembered.


## 1. Setup — reuse the MCP agent from Module 1.3

Reflection is *additive*: we wrap the Actor from Notebook 4, we don't rebuild it. The same `CodeAgent` with `search_arxiv` + `run_ising_simulation` will be our agent under test.

As before, `ToolCollection.from_mcp` is a context manager; we enter it manually so later cells can reuse the connection, and we exit it cleanly at the end of the notebook.


In [ ]:
import os
import sys
import warnings

# Silence a known-harmless LiteLLM/Pydantic v2 schema warning on Ollama.
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

from mcp import StdioServerParameters
from smolagents import ToolCollection, CodeAgent, LiteLLMModel

SERVER_PARAMS = StdioServerParameters(
    command=sys.executable,
    args=["-m", "mcp_server.physics_tools_server"],
    env={**os.environ, "PYTHONPATH": ".."},
    cwd="..",
)

_mcp_cm = ToolCollection.from_mcp(SERVER_PARAMS)
tc = _mcp_cm.__enter__()

MODEL = "qwen2.5:7b"

actor_model = LiteLLMModel(
    model_id=f"ollama_chat/{MODEL}",
    api_base="http://localhost:11434",
    api_key="ollama",
    temperature=0.0,
)

actor = CodeAgent(
    tools=[*tc.tools],
    model=actor_model,
    additional_authorized_imports=["math", "json", "numpy"],
    max_steps=8,
)

print("Actor ready. Tools:", list(actor.tools.keys()))


## 2. The Critic — one function, no agent, no tools

The Critic is deliberately *not* another `CodeAgent`. It has no tools, no multi-step reasoning, no looping. It is a single LLM call with a system prompt and a strict output schema. Keeping it this small matters for three reasons:

1. It runs in a few seconds.
2. Its behaviour is easy to eyeball — the whole "Critic" is ~20 lines of Python.
3. It composes with *any* producer, not just our `CodeAgent`. Replace the Actor with a different framework and the Critic still works.

The system prompt is the outline's "senior physicist reviewing a junior" framing. We ask the Critic to return a JSON object with `verdict` ∈ {`accept`, `revise`}, a short `feedback` message, and — if it's rejecting — a one-line `lesson` that summarises what the Actor should remember next time. That `lesson` is what the Reflexion memory will store in §5.


In [ ]:
import json as _json
import re
from litellm import completion

CRITIC_SYSTEM = """You are a senior physicist reviewing a junior colleague's answer to a physics question. Check for: (1) mathematical correctness, (2) consistency between theoretical predictions and any simulation results quoted, (3) whether the junior has actually answered the question that was asked (no hand-waving, no raw-JSON dumps), and (4) any signs of hallucination. Be specific.

Respond with a JSON object and nothing else, of the form:

  {"verdict": "accept" | "revise",
   "feedback": "<one or two sentences explaining the verdict>",
   "lesson":   "<if verdict==revise, a single short imperative sentence the junior should remember next time; else empty string>"}
"""

def critique(task: str, answer: str, *, model: str = f"ollama_chat/{MODEL}") -> dict:
    """One LLM call; returns {"verdict", "feedback", "lesson"}."""
    user = (
        f"Task given to the junior:\n{task}\n\n"
        f"Junior's answer:\n{answer}\n\n"
        "Return only the JSON object specified in the system prompt."
    )
    resp = completion(
        model=model,
        api_base="http://localhost:11434",
        api_key="ollama",
        messages=[
            {"role": "system", "content": CRITIC_SYSTEM},
            {"role": "user",   "content": user},
        ],
        temperature=0.0,
    )
    raw = resp["choices"][0]["message"]["content"]
    # The model sometimes wraps JSON in a code fence -- strip it pragmatically.
    m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if m is None:
        return {"verdict": "accept",
                "feedback": f"Critic produced non-JSON output: {raw[:120]}...",
                "lesson": ""}
    try:
        parsed = _json.loads(m.group(0))
    except _json.JSONDecodeError:
        return {"verdict": "accept",
                "feedback": f"Critic JSON did not parse: {raw[:120]}...",
                "lesson": ""}
    parsed.setdefault("verdict", "accept")
    parsed.setdefault("feedback", "")
    parsed.setdefault("lesson", "")
    return parsed


### 2.1 Smoke-test the Critic on an obviously wrong answer

Before wiring the loop, confirm the Critic can tell good from bad on a trivially broken answer. The exact critical temperature of the 2D Ising model (J = 1, k_B = 1) is $T_c = 2/\ln(1+\sqrt{2}) \approx 2.269$; an answer of "42 kelvin" is unambiguously wrong (wrong value *and* wrong units), so we expect `verdict == "revise"`.


In [ ]:
result = critique(
    task="What is the exact critical temperature of the 2D Ising model (J=1, k_B=1)?",
    answer="T_c is 42 kelvin.",
)
print("verdict :", result["verdict"])
print("feedback:", result["feedback"])
print("lesson  :", result["lesson"])


## 3. The reflection loop

`run_with_reflection` is the whole "self-correction" mechanism: *actor → critic → if revise, retry once with the feedback appended*. We cap at `max_retries=2` — the outline's choice, and also the empirical sweet spot (more rounds rarely help; they just burn context).

Two implementation notes:

- **We call the agent directly, not its `run(task)` convenience wrapper.** That's fine because `CodeAgent.run(task)` just threads `task` into the initial user prompt; for reflection we need to append the Critic's feedback, so we construct the revised prompt ourselves.
- **We reset the agent's memory between rounds.** Small models otherwise accumulate state from the previous attempt (including the wrong reasoning), which tends to entrench the mistake instead of correcting it.


In [ ]:
def run_with_reflection(
    agent,
    task: str,
    *,
    max_retries: int = 2,
    memory_lessons: list[str] | None = None,
    verbose: bool = True,
) -> dict:
    """Run the agent with a Reflection loop. Returns a dict with the final
    answer, the transcript of (answer, critique) pairs, and the list of
    lessons distilled from any rejections along the way.
    """
    lessons_this_task: list[str] = []
    transcript: list[dict] = []

    # If the caller passed in persistent memory, prepend it to the prompt.
    prompt = task
    if memory_lessons:
        bullets = "\n".join(f"- {l}" for l in memory_lessons)
        prompt = (
            "Lessons from prior tasks (apply them if relevant):\n"
            f"{bullets}\n\n"
            f"Task:\n{task}"
        )

    for attempt in range(max_retries + 1):  # 1 initial + max_retries revisions
        if verbose:
            print(f"\n=== Attempt {attempt + 1} ===")

        agent.memory.reset()  # fresh slate each round
        answer = agent.run(prompt)

        if verbose:
            print("\n--- Actor answer ---")
            print(str(answer)[:500] + ("..." if len(str(answer)) > 500 else ""))

        verdict = critique(task, str(answer))
        transcript.append({"answer": str(answer), "critique": verdict})

        if verbose:
            print(f"\n--- Critic: {verdict['verdict']} ---")
            print(verdict["feedback"])

        if verdict["verdict"] == "accept":
            return {"final_answer": str(answer),
                    "accepted_on_attempt": attempt + 1,
                    "transcript": transcript,
                    "lessons": lessons_this_task}

        # Rejected -- store the lesson and build a revised prompt.
        if verdict["lesson"]:
            lessons_this_task.append(verdict["lesson"])

        prompt = (
            f"{task}\n\n"
            f"Your previous answer was rejected by the reviewer with this feedback:\n"
            f"  {verdict['feedback']}\n\n"
            "Revise the answer to address that feedback, and explicitly mention "
            "the numerical comparison to Onsager's exact result where relevant."
        )

    return {"final_answer": str(answer),
            "accepted_on_attempt": None,  # exhausted retries
            "transcript": transcript,
            "lessons": lessons_this_task}


## 4. Demo — the finite-size-effect catch

Ask the agent to estimate $T_c$ from the **specific-heat peak** on an 8×8 lattice. That's a deliberately small lattice: the apparent $T_c$ is shifted a few percent above the infinite-system value of 2.269 because the correlation length saturates at the system size.

We spell the method out in the prompt so qwen2.5:7b can execute it reliably (if you want to see the small model struggle, strip those hints and re-run). The *physics* point of the demo is that without reflection, the agent will happily report e.g. 2.4 as $T_c$ without acknowledging the shift. With reflection, the Critic catches the discrepancy with Onsager and asks for a revision.

> **Why the specific-heat peak?** At a second-order phase transition, $C_V$ diverges on the infinite lattice and develops a sharp, size-limited peak on a finite one. That peak position is a standard finite-size estimator of $T_c$; in 2D Ising the shift is $O(1/L)$ so 8×8 is meaningfully biased, 32×32 noticeably less so, 64×64 essentially converged.


In [ ]:
TASK_TC = (
    "Estimate the critical temperature T_c of the 2D Ising model on a "
    "LATTICE_SIZE x LATTICE_SIZE lattice using the specific-heat peak. "
    "Call `run_ising_simulation` at each of T = 2.10, 2.20, 2.27, 2.35, 2.45, "
    "with lattice_size=LATTICE_SIZE, num_steps=3000, algorithm='wolff', seed=1. "
    "For each run, parse the returned JSON string with json.loads and extract "
    "the `specific_heat` field. Report as your T_c estimate the temperature at "
    "which specific_heat is largest. Also state numerically how far your "
    "estimate is from Onsager's exact result 2.269."
)

task_8 = TASK_TC.replace("LATTICE_SIZE", "8")

result_8 = run_with_reflection(actor, task_8, max_retries=2)

print("\n================ SUMMARY ================")
print("Accepted on attempt:", result_8["accepted_on_attempt"])
print("Lessons distilled :", result_8["lessons"])
print("\nFinal answer:\n", result_8["final_answer"])


## 5. Reflexion — persistent memory across tasks

Reflection, by itself, is amnesiac: the next task starts from the same blank slate. **Reflexion** fixes that with a tiny piece of infrastructure: a list of lesson-strings on disk, read at the start of every task, appended to at the end.

Our `ReflexionMemory` is about as minimal as it gets. Under the hood it is one JSON file with one list of strings. Methods:

- `lessons() -> list[str]` — what we've learned so far.
- `add(lesson: str)` — store one more, with de-duplication so the file doesn't grow unbounded if the Critic keeps flagging the same issue.
- `clear()` — wipe the slate; we'll use this to start the demo clean.

Dropping it into the reflection loop is one line: we pass `memory.lessons()` as `memory_lessons` to `run_with_reflection`. When the Actor is rebuilt for a new task, its prompt now carries every lesson the Critic has ever distilled.


In [ ]:
from pathlib import Path

MEMORY_PATH = Path("../data/reflexion_memory.json")

class ReflexionMemory:
    """JSON-file-backed list of lesson-strings. Deliberately tiny."""

    def __init__(self, path: Path = MEMORY_PATH):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        if not self.path.exists():
            self.path.write_text("[]")

    def lessons(self) -> list[str]:
        return json.loads(self.path.read_text())

    def add(self, lesson: str) -> None:
        if not lesson:
            return
        cur = self.lessons()
        if lesson in cur:  # naive dedup
            return
        cur.append(lesson)
        self.path.write_text(json.dumps(cur, indent=2))

    def clear(self) -> None:
        self.path.write_text("[]")

memory = ReflexionMemory()
memory.clear()
print(f"Memory initialised at {memory.path.resolve()}, lessons: {memory.lessons()}")


### 5.1 Task 1 revisited — now feeding rejections into memory

Same 8×8 task as §4, but this time any `lesson` produced by the Critic is written to the Reflexion memory file before we move on. After this cell, `data/reflexion_memory.json` will contain one or two distilled lessons — open the file in your editor and read them.


In [ ]:
result1 = run_with_reflection(
    actor,
    task_8,
    max_retries=2,
    memory_lessons=memory.lessons(),  # currently []
)

for l in result1["lessons"]:
    memory.add(l)

print("\n================ AFTER TASK 1 ================")
print("Lessons stored:")
for l in memory.lessons():
    print(f"  - {l}")


### 5.2 Task 2 — same question, bigger lattice, *with memory*

We now ask the same kind of question on a 32×32 lattice. Because `memory.lessons()` is non-empty, the Actor sees the distilled warnings from Task 1 in its prompt. Expected behaviour: the agent *preemptively* mentions finite-size effects and how the 32×32 estimate compares to Onsager's 2.269, without the Critic having to intervene again. In other words, the Reflection round should `accept` on attempt 1.

(If you watch qwen2.5:7b closely you'll see it occasionally ignore the lessons anyway — small models are not perfectly steerable. That's a useful dose of honesty about the limits of prompting as a control knob, and it motivates the more structural multi-agent designs in Day 2.)


In [ ]:
task_32 = TASK_TC.replace("LATTICE_SIZE", "32")

result2 = run_with_reflection(
    actor,
    task_32,
    max_retries=2,
    memory_lessons=memory.lessons(),
)

for l in result2["lessons"]:
    memory.add(l)

print("\n================ AFTER TASK 2 ================")
print("Accepted on attempt:", result2["accepted_on_attempt"])
print("Lessons now stored:")
for l in memory.lessons():
    print(f"  - {l}")


### 5.3 Task 3 — a theoretical check that *uses* the lesson

The 2D Ising model has critical exponent $\beta = 1/8$, meaning $|M| \sim (T_c - T)^{1/8}$ as $T \to T_c^-$. We ask the agent to measure $|M|$ at one temperature safely below $T_c$ on a lattice that is "large enough not to bias the answer." If the Reflexion memory is doing its job, the agent will pick $L \geq 32$ on its own, compute $|M|$, and compare it to the theoretical value $(1 - \sinh(2/T)^{-4})^{1/8}$ that follows from Onsager's exact solution (valid for $T < T_c$ on the infinite lattice — so the agent should note the lattice is only approximately infinite).


In [ ]:
TASK_BETA = (
    "At T = 2.0 (below T_c = 2.269), Onsager's exact solution gives "
    "|M|(T) = (1 - sinh(2/T)^(-4))^(1/8) for the 2D Ising model on the "
    "infinite lattice. Pick a lattice size appropriate for a reliable "
    "Monte Carlo estimate, run `run_ising_simulation` at T=2.0 with "
    "algorithm='wolff' and num_steps=5000, json.loads the result, read "
    "`magnetization_mean`, and compare it numerically to Onsager's value. "
    "State the percentage difference and whether it is consistent with "
    "finite-size effects."
)

result3 = run_with_reflection(
    actor,
    TASK_BETA,
    max_retries=2,
    memory_lessons=memory.lessons(),
)

for l in result3["lessons"]:
    memory.add(l)

print("\n================ AFTER TASK 3 ================")
print("Accepted on attempt:", result3["accepted_on_attempt"])
print("Final answer:\n", result3["final_answer"][:600])
print("\nFinal memory:")
for l in memory.lessons():
    print(f"  - {l}")


## 6. Clean shutdown

In [ ]:
_mcp_cm.__exit__(None, None, None)
print("MCP server stopped.")


## 7. Key takeaway + exercises

**What we added, concretely.**

| Component | Lines of code | Role |
|---|---|---|
| `critique(task, answer)` | ~25 | Judge — single LLM call, JSON verdict |
| `run_with_reflection(agent, task)` | ~30 | Loop — at most 3 rounds (1 + 2 retries) |
| `ReflexionMemory` | ~20 | File-backed list of lesson-strings |

That's it. ~75 lines, no new dependencies, no changes to the MCP server or the underlying `CodeAgent`. **Reflection and Reflexion are pure scaffolding.** The model hasn't been fine-tuned; the tools haven't changed; only the loop around the model is richer.

**Why this matters pedagogically.** Module 1.3 ended with the observation that the agent took shortcuts (returning raw JSON, skipping English summaries). You might have been tempted to "fix" that by hand-engineering a more coercive prompt. Reflection makes the coercion *adaptive*: the Critic notices what's missing *on this specific task* and asks for exactly that. Reflexion then generalises: once a class of mistake has been seen, it is avoided preemptively on the next task.

**Where this breaks.**
- *Judge-as-LLM is not a judge-in-fact.* The Critic itself can hallucinate, especially on subtle physics. Treat it as a second opinion, not ground truth. The Day 2 multi-agent design replaces "one agent's Critic" with "a specialised Theorist agent whose job is verification" — same pattern, more accountability.
- *Memory drift.* If the Critic's lessons contradict each other (or the user's actual intent), the memory becomes noise. A real deployment needs a curation step; ours has none. That's what Exercise 3 explores.

### Exercises

1. **Turn reflection off.** Set `max_retries=0` and re-run Task 1. You should see the Actor report a biased $T_c$ without any mention of finite-size effects. That is the baseline "confidently wrong" behaviour Reflection is designed to catch.

2. **A harder Critic.** Modify `CRITIC_SYSTEM` to require that any numerical answer be accompanied by (a) a comparison to a theoretical or literature value, and (b) an explicit error bar. Re-run Task 3. Does the Actor improve, or does it start parroting error bars out of thin air? Either way, you'll learn something about the limits of LLM-as-judge.

3. **Memory curation.** After running Tasks 1–3, open `data/reflexion_memory.json` and ask: do the stored lessons actually generalise? Are any of them specific enough to hurt (e.g. "always use L=32" — what if the student genuinely wants small-L behaviour for a finite-size-scaling study)? Write a one-cell helper that lets you edit the memory and re-run Task 3 with the curated version.

4. **Swap the Critic model.** Keep the Actor as `qwen2.5:7b` but run the Critic as `llama3.1:8b`. Different models have different failure modes; a cross-model Critic catches some failures that a same-model Critic would miss. This is a cheap, practical reliability lever.

---

**Next up — Module 1.5:** We bring everything together — RAG + MCP tools + Reflection + Reflexion memory — into a single *Physics Research Assistant*, the Day 1 climax. After that, Day 2 opens by asking: what if, instead of one agent trying to be good at everything, we had several specialised agents collaborating?
